# 09 bis — Capturer des images réelles

Capture visuelle depuis la mémoire partagée du plugin `sensors_par`. **Lance le jeu et active `screen_capture_enable`**. Les images sont stockées sans étiquette : elles ne constituent pas automatiquement un jeu de données supervisé.

In [ ]:
from pathlib import Path
import sys
CWD=Path.cwd().resolve()
REPO=next((p for p in [CWD,*CWD.parents] if (p/'assetto_corsa_gym'/'assetto_corsa_gym'/'AssettoCorsaEnv').is_dir()),None)
if REPO is None:
    raise FileNotFoundError('Ouvre le depot IA_projet_Ouarti_ds_github dans VS Code : son dossier assetto_corsa_gym doit exister.')
PROJECT=REPO/'OUARTI'
if not PROJECT.is_dir():
    raise FileNotFoundError('Dossier OUARTI absent : extraire OUARTI a la racine du depot.')
GYM_ROOT=REPO/'assetto_corsa_gym'/'assetto_corsa_gym'
print('Depot :',REPO)
print('Notebooks :',PROJECT)
print('Python :',sys.version.split()[0],sys.executable)


In [ ]:
import socket, json, ast, time
HOST='127.0.0.1'
PORT_UDP=2345
PORT_TCP=2347

def management(message,timeout=3):
    with socket.create_connection((HOST,PORT_TCP),timeout=timeout) as s:
        s.settimeout(timeout);s.sendall(message.encode('ascii'))
        data=b''
        end=time.monotonic()+timeout
        while time.monotonic()<end and len(data)<1024*1024:
            try:chunk=s.recv(65536)
            except socket.timeout:break
            if not chunk:break
            data+=chunk
            try:return json.loads(data)
            except (json.JSONDecodeError, UnicodeDecodeError):pass
            try:return ast.literal_eval(data.decode('utf8'))
            except (ValueError,SyntaxError,UnicodeDecodeError):pass
        if data:
            raise RuntimeError('Reponse TCP incomplete : '+repr(data[:160]))
    raise RuntimeError('Pas de reponse sur le port management')

def open_telemetry(timeout=4):
    s=socket.socket(socket.AF_INET,socket.SOCK_DGRAM);s.settimeout(.4)
    s.connect((HOST,PORT_UDP))
    end=time.monotonic()+timeout
    while time.monotonic()<end:
        s.send(b'connect')
        try:response=s.recv(2**18)
        except socket.timeout:continue
        if response==b'identified':return s
    s.close()
    raise TimeoutError('Pas de confirmation UDP sur 2345 : verifier jeu en piste et plugin sensors_par.')

def next_packet(s,timeout=3):
    end=time.monotonic()+timeout
    while time.monotonic()<end:
        try:message=s.recv(2**18)
        except socket.timeout:continue
        if message==b'disconnect':raise ConnectionError('Le jeu a ferme le flux UDP')
        if message==b'identified':continue
        raw=json.loads(message)
        if not isinstance(raw,dict):continue
        raw['_capture_wall_time']=time.time()
        return raw
    raise TimeoutError('Aucun echantillon recu.')

def close_telemetry(s):
    if s is not None:
        try:s.send(b'disconnect')
        except OSError:pass
        s.close()


In [ ]:
config=management('get_config')
static=management('get_static_info')
print('Plugin :')
for key in ['ego_sampling_freq','sampling_freq','vjoy_executed_by_server','screen_capture_enable','color_mode','final_image_width','final_image_height']:
    print(' ',key,':',config.get(key))
print('Session :',static.get('TrackName'),static.get('TrackConfiguration'), '/', static.get('CarName'))


In [ ]:
from datetime import datetime
import sys,time,json,numpy as np
from PIL import Image
if not config.get('screen_capture_enable'):
    raise RuntimeError('Activer screen_capture_enable dans le plugin installe sur le jeu.')
if sys.platform!='win32':raise RuntimeError('Memoire partagee de cette version prevue pour Windows.')
plugin_path=GYM_ROOT/'AssettoCorsaPlugin'/'plugins'/'sensors_par'
sys.path.insert(0,str(plugin_path))
from dual_buffer import GrabberSharedMemoryDualBuffer
camera=GrabberSharedMemoryDualBuffer(int(config['final_image_width']),int(config['final_image_height']),str(config['color_mode']))
DUREE=10
DELAI=0.3
output=PROJECT/'datasets'/'vision'/'images'
output.mkdir(parents=True,exist_ok=True)
t0=time.monotonic();saved=0;seen=set()
while time.monotonic()-t0<DUREE:
    if hasattr(camera,'get_timed_image'):
        try:seq,stamp,frame=camera.get_timed_image()
        except RuntimeError:
            time.sleep(DELAI);continue
        if seq in seen or stamp<=0:
            time.sleep(DELAI);continue
        seen.add(seq)
    else:
        _,_,frame=camera.get_active_image()
        stamp=time.time()
        seq=saved
    if config.get('color_mode')=='bgr':
        frame=frame[:,:,::-1]
    if not np.any(frame):
        time.sleep(DELAI);continue
    Image.fromarray(frame).save(output/('capture_%06d.png'%seq))
    saved+=1
    time.sleep(DELAI)
print('Images capturees :',saved,'dossier :',output)
if saved==0:raise RuntimeError('Aucune image valide : verifier le worker capture et la config.')
